<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M01/M01_Lab1_API_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🚀 M01 Lab 1 — Your First LLM API Call</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~10 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Set up API keys securely using <b>Google Colab Secrets</b></li>
    <li>Make your first API call to OpenAI's <code>gpt-4.1-mini</code></li>
    <li>Understand the <b>anatomy of a ChatCompletion response</b> — content, roles, and usage</li>
    <li>Build <b>multi-turn conversations</b> by passing message history</li>
    <li>Explore a real-world use case: building a simple customer support assistant</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai tiktoken
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from openai import OpenAI
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
# If it works, you'll see a confirmation message below
client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ The Chat Completions API</h2>
</div>

The core of every LLM application is the **Chat Completions API**. It works like a conversation:

- You send a list of **messages** (the conversation so far)
- Each message has a **role** (`system`, `user`, or `assistant`)
- The model returns a new `assistant` message as a response

| Role | Purpose | Example |
|------|---------|--------|
| `system` | Sets the assistant's persona, rules, and constraints | *"You are a helpful tutor who explains things simply."* |
| `user` | Your input / question | *"What is generative AI?"* |
| `assistant` | The model's response (also used for conversation history) | *"Generative AI refers to..."* |

Let's make our first call:

In [ ]:
# ============================================================
# 🚀 Your First API Call
# ============================================================

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is generative AI? Explain in 2-3 sentences."}
    ]
)

# Pretty-print the response
show_response(response)

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> The <code>system</code> message is optional but powerful. It shapes <i>how</i> the model responds — tone, format, length, even personality. We'll explore this deeply in M02.
</div>

Notice the response also shows **token usage**. Tokens are the "currency" of LLM APIs — we'll cover them in detail in Lab 2.

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Response Anatomy</h2>
</div>

The response from OpenAI is a **structured object**, not just a string. Understanding its structure is essential for building real applications.

Let's inspect every part of it:

In [ ]:
# ============================================================
# 🔍 Inspecting the Response Object
# ============================================================

print("📋 Model used:      ", response.model)
print("🆔 Response ID:     ", response.id)
print()

# The actual reply
message = response.choices[0].message
print("💬 Role:            ", message.role)
print("📝 Content:         ", message.content)
print()

# Token usage — how much this call "cost" in tokens
usage = response.usage
print("📊 --- Token Usage ---")
print(f"   Prompt tokens:     {usage.prompt_tokens}")
print(f"   Completion tokens: {usage.completion_tokens}")
print(f"   Total tokens:      {usage.total_tokens}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaways:</b>
  <ul style="margin: 6px 0 0;">
    <li><code>response.choices[0].message.content</code> — this is the actual text reply you'll use 99% of the time</li>
    <li><code>response.usage</code> — tells you how many tokens were consumed (important for cost tracking)</li>
    <li>The model can return <b>multiple choices</b> (via the <code>n</code> parameter), but typically we just use <code>choices[0]</code></li>
  </ul>
</div>

### Trying Different Questions

Let's see how the model handles different types of questions — factual, creative, and analytical:

In [ ]:
# ============================================================
# 🎯 Different Question Types
# ============================================================

questions = [
    ("Factual",    "What are the three main types of machine learning?"),
    ("Creative",   "Write a 2-line poem about data science."),
    ("Analytical", "What are the pros and cons of using LLMs in healthcare? Give 2 of each."),
]

for q_type, question in questions:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": question}],
        max_tokens=150,
    )
    print(f"\n{'='*60}")
    print(f"📌 {q_type}: {question}")
    print(f"{'='*60}")
    print(f"\n{r.choices[0].message.content.strip()}")
    print(f"\n   📊 Tokens used: {r.usage.total_tokens}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Multi-Turn Conversations</h2>
</div>

LLMs are **stateless** — they don't remember previous calls. To create a conversation, you must pass the **full message history** each time.

This is how ChatGPT works under the hood: every time you send a message, the entire conversation is re-sent to the API.

In [ ]:
# ============================================================
# 💬 Building a Multi-Turn Conversation
# ============================================================

# Start with a system message and first question
messages = [
    {"role": "system", "content": "You are a knowledgeable AI tutor. Keep answers concise (2-3 sentences)."},
    {"role": "user", "content": "What is a neural network?"}
]

# Turn 1
r1 = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)
reply1 = r1.choices[0].message.content
print("🧑 User: What is a neural network?")
print(f"🤖 Assistant: {reply1}\n")

# Add the assistant's reply to history, then ask a follow-up
messages.append({"role": "assistant", "content": reply1})
messages.append({"role": "user", "content": "How is that different from a traditional algorithm?"})

# Turn 2 — the model now has context from Turn 1
r2 = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)
reply2 = r2.choices[0].message.content
print("🧑 User: How is that different from a traditional algorithm?")
print(f"🤖 Assistant: {reply2}\n")

# Turn 3 — reference something from earlier
messages.append({"role": "assistant", "content": reply2})
messages.append({"role": "user", "content": "Give me a real-world example of what you just described."})

r3 = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)
print("🧑 User: Give me a real-world example of what you just described.")
print(f"🤖 Assistant: {r3.choices[0].message.content}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Did You Notice?</b> In Turn 3, the model understood "what you just described" because we passed the full conversation history. Without that context, it would have no idea what we were referring to. This is fundamental to how all chat-based AI apps work.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Real-World Example: Customer Support Assistant</h2>
</div>

Let's build something practical — a simple customer support assistant for a fictional company. This shows how the `system` message shapes behavior in a real scenario.

In [ ]:
# ============================================================
# 🏢 Customer Support Assistant
# ============================================================

SYSTEM_PROMPT = """You are a customer support agent for TechStore, an online electronics retailer.

Rules:
- Be friendly and professional
- Keep responses under 3 sentences
- If you don't know something, say "Let me connect you with a specialist"
- Never discuss competitor products
- Always end with "Is there anything else I can help with?"
"""

# Simulate a customer conversation
customer_messages = [
    "Hi, I ordered a laptop last week and it hasn't arrived yet. Order #12345.",
    "Can you check when it will be delivered?",
    "Also, is the MacBook Pro better than what I ordered?",
]

conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

for msg in customer_messages:
    conversation.append({"role": "user", "content": msg})

    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=conversation,
        max_tokens=150,
    )
    reply = r.choices[0].message.content
    conversation.append({"role": "assistant", "content": reply})

    print(f"🧑 Customer: {msg}")
    print(f"🤖 Agent:    {reply}")
    print()

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 What to Notice:</b>
  <ul style="margin: 6px 0 0;">
    <li>The agent <b>follows all the rules</b> from the system prompt — length, tone, and the competitor deflection</li>
    <li>When asked about MacBook (a competitor), it should redirect rather than answer</li>
    <li>The agent <b>maintains context</b> across the conversation — it remembers the order number</li>
  </ul>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is a token in the context of LLMs?",
        "options": [
            "A security credential for API access",
            "A piece of text — typically a word, subword, or character",
            "A type of neural network layer",
            "A unit of GPU memory"
        ],
        "answer": 1,
        "explanation": "Tokens are the basic units that LLMs process. A word like 'hello' is typically one token, while longer or uncommon words get split into multiple subword tokens."
    },
    {
        "q": "Why do we pass the full message history in multi-turn conversations?",
        "options": [
            "To increase the API cost",
            "Because LLMs are stateless — they don't remember previous calls",
            "Because the API requires at least 5 messages",
            "To make the response faster"
        ],
        "answer": 1,
        "explanation": "Each API call is independent. The model has no memory of previous calls, so you must include the full conversation history for context."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build a Travel Advisor Bot

Complete the code below to create a travel advisor assistant. Replace each `-----` with the correct value.

**Your bot should:**
- Act as a friendly travel advisor
- Recommend destinations based on preferences
- Keep responses concise (under 4 sentences)

In [ ]:
# ============================================================
# 🔧 Exercise: Travel Advisor Bot
# ============================================================
# Replace each ----- with the correct value

# Step 1: Define the system message for your travel advisor
travel_system = """You are a friendly travel advisor. -----"""
# 👆 YOUR CODE: Add 2-3 rules (e.g., keep responses concise, suggest budget-friendly options)

# Step 2: Create the messages list
travel_messages = [
    {"role": "-----", "content": travel_system},           # What role is the system message?
    {"role": "user", "content": "I want a relaxing beach vacation for under $2000. Any suggestions?"}
]

# Step 3: Make the API call
travel_response = client.chat.completions.create(
    model="-----",                                           # Which model should we use?
    messages=travel_messages,
    max_tokens=200,
)

# Step 4: Extract and print the reply
reply = travel_response.-----[0].message.-----              # How do we access the content?
print(f"🌴 Travel Advisor: {reply}")
print(f"\n📊 Tokens used: {travel_response.usage.total_tokens}")

In [ ]:
# ============================================================
# 🔧 Exercise (continued): Add a follow-up question
# ============================================================
# Now add the assistant's reply to the conversation and ask a follow-up

travel_messages.append({"role": "-----", "content": reply})   # What role is the assistant reply?
travel_messages.append({"role": "user", "content": "-----"})   # YOUR CODE: Ask a follow-up question

travel_r2 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=travel_messages,
    max_tokens=200,
)

print(f"🌴 Travel Advisor: {travel_r2.choices[0].message.content}")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the travel advisor follow the rules you set in the system message? _[Your answer]_

2. Did it remember context from the first question in its follow-up response? _[Your answer]_

3. What happens if you change the system message to a different persona (e.g., luxury travel agent)? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try modifying the customer support assistant above:</p>
  <ol style="font-size: 14px;">
    <li>Change the company from TechStore to a restaurant — how does the system prompt change?</li>
    <li>Add a rule: "Always respond in exactly 2 sentences" — does the model follow it?</li>
    <li>Try asking the model something completely off-topic — how does it handle it?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've made your first LLM API calls, explored response objects, and built a multi-turn assistant.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">client.chat.completions.create()</code> is the core API call</li>
      <li>Messages have <b>roles</b>: <code style="color: #90caf9;">system</code>, <code style="color: #90caf9;">user</code>, <code style="color: #90caf9;">assistant</code></li>
      <li>LLMs are <b>stateless</b> — pass full conversation history for context</li>
      <li>The <code style="color: #90caf9;">system</code> message is your main tool for controlling model behavior</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M01 Lab 2: Model Parameters &amp; Token Economics</p>
</div>